In [3]:
import time

class PDController:
    """
    A basic implementation of a PD (Proportional-Derivative) Controller.
    """
    def __init__(self, Kp, Kd, setpoint, sample_time=0.01, output_limits=(None, None)):
        """
        Initialize the PD controller.

        Args:
            Kp (float): Proportional gain.
            Kd (float): Derivative gain.
            setpoint (float): The desired target value.
            sample_time (float): The time interval (in seconds) between controller updates.
                                 Ensures consistent derivative calculation.
            output_limits (tuple): A tuple (min_output, max_output) to constrain the controller output.
                                   Use None for no limit.
        """
        # --- Gains ---
        self.Kp = Kp
        self.Kd = Kd

        # --- Setpoint ---
        self.setpoint = setpoint

        # --- Timing ---
        self.sample_time = sample_time # Time interval between updates
        self.last_time = time.time()   # Timestamp of the last update

        # --- Internal State ---
        self.last_error = 0.0       # Error from the previous update cycle
        self.last_output = 0.0      # Store the last output value

        # --- Output Limits ---
        self.min_output, self.max_output = output_limits

        # --- Set initial time ---
        self.reset() # Initialize time and error properly

    def update(self, current_value):
        """
        Calculate the PD output value for a given process variable input.

        Args:
            current_value (float): The current measured value of the process variable.

        Returns:
            float: The calculated controller output.
        """
        current_time = time.time()
        delta_time = current_time - self.last_time

        # --- Only update if sample time has passed ---
        if delta_time < self.sample_time:
            # Return the last calculated output if called too soon
            return self.last_output

        # --- Calculate Error ---
        error = self.setpoint - current_value

        # --- Proportional Term ---
        proportional_term = self.Kp * error

        # --- Derivative Term ---
        # Calculate delta_error based on the change in error since the last update
        delta_error = error - self.last_error
        derivative_term = 0.0
        if delta_time > 0: # Avoid division by zero
            # Calculate derivative based on error change rate
            derivative_term = self.Kd * (delta_error / delta_time)

        # --- Calculate Total Output ---
        # Sum only Proportional and Derivative terms
        output = proportional_term + derivative_term

        # --- Apply Output Limits ---
        if self.min_output is not None:
            output = max(self.min_output, output)
        if self.max_output is not None:
            output = min(self.max_output, output)

        # --- Update State for Next Iteration ---
        self.last_error = error
        self.last_time = current_time
        self.last_output = output # Store the calculated output

        return output

    def set_setpoint(self, new_setpoint):
        """Update the setpoint."""
        self.setpoint = new_setpoint
        # Reset last error when setpoint changes to avoid large derivative spike
        self.last_error = 0.0

    def set_gains(self, Kp, Kd):
        """Update the gains."""
        self.Kp = Kp
        self.Kd = Kd

    def reset(self):
        """Resets the internal state of the PD controller."""
        self.last_error = 0.0
        self.last_time = time.time() # Reset timer
        self.last_output = 0.0 # Reset last output


# --- Example Usage (Conceptual) ---

# Example: Control temperature of a simulated system (similar to PID example)
# Note: With only PD, expect some steady-state error depending on the system dynamics and gains.
KP = 1.5
KD = 0.1 # Derivative gain helps dampen oscillations
SETPOINT = 100.0  # Target temperature
SAMPLE_TIME = 0.1 # Update every 0.1 seconds
OUTPUT_LIMITS = (0, 200) # Example limits

# Initialize the controller
pd = PDController(KP, KD, SETPOINT, sample_time=SAMPLE_TIME, output_limits=OUTPUT_LIMITS)

# --- Simulation Loop ---
current_temperature = 20.0 # Initial temperature
simulation_duration = 10 # seconds
start_time = time.time()

print(f"Starting PD simulation. Target: {SETPOINT}°C, Initial: {current_temperature}°C")
print("-" * 30)
print("Time (s) | Temp (°C) | Error (°C) | Output")
print("-" * 30)

while time.time() - start_time < simulation_duration:
    # --- Get controller output ---
    control_output = pd.update(current_temperature)

    # --- Simulate the system's response (same simple model as before) ---
    heat_gain = control_output * 0.05
    heat_loss = (current_temperature - 20) * 0.01
    temperature_change = (heat_gain - heat_loss) * pd.sample_time

    current_temperature += temperature_change

    # --- Print status ---
    elapsed_time = time.time() - start_time
    error = pd.setpoint - current_temperature
    print(f"{elapsed_time:8.2f} | {current_temperature:9.2f} | {error:10.2f} | {control_output:6.2f}")

    # --- Wait for next sample time ---
    time.sleep(max(0, pd.sample_time - (time.time() - pd.last_time)))


print("-" * 30)
print("Simulation finished.")
print(f"Final Temperature: {current_temperature:.2f}°C (Note potential steady-state error)")



Starting PD simulation. Target: 100.0°C, Initial: 20.0°C
------------------------------
Time (s) | Temp (°C) | Error (°C) | Output
------------------------------
    0.00 |     20.00 |      80.00 |   0.00
    0.10 |     21.00 |      79.00 | 199.85
    0.20 |     21.59 |      78.41 | 117.50
    0.30 |     22.17 |      77.83 | 117.04
    0.40 |     22.75 |      77.25 | 116.16
    0.50 |     23.32 |      76.68 | 115.30
    0.60 |     23.89 |      76.11 | 114.44
    0.70 |     24.45 |      75.55 | 113.60
    0.80 |     25.01 |      74.99 | 112.75
    0.90 |     25.57 |      74.43 | 111.92
    1.00 |     26.12 |      73.88 | 111.09
    1.10 |     26.66 |      73.34 | 110.27
    1.20 |     27.20 |      72.80 | 109.46
    1.30 |     27.74 |      72.26 | 108.65
    1.40 |     28.27 |      71.73 | 107.85
    1.50 |     28.80 |      71.20 | 107.06
    1.60 |     29.32 |      70.68 | 106.28
    1.70 |     29.84 |      70.16 | 105.50
    1.80 |     30.35 |      69.65 | 104.72
    1.90 |     30.86 